# Kaggle submission (OFFLINE) — Akkadian → English (ByT5)

Code-competition без интернета: Kaggle приватно перезапускает выбранную версию со
скрытым тестом и забирает `submission.csv` из Output. Интернет **выключен**, поэтому
ноутбук самодостаточный: код инференса (нормализация + MBR-chrF) вшит ниже, модели
грузятся из подключённых Kaggle Dataset. Наш пакет / HF Hub / sacrebleu не нужны.

Финальная конфигурация — **ансамбль seed13 + seed42** (по 4 beam-кандидата с каждой,
выбор консенсуса через MBR-chrF). Для одиночной модели оставь один путь в `MODEL_DIRS`.

**Перед сабмитом:**
1. **Input → + Add Input**: соревнование *Deep Past* и **два датасета** с моделями
   (по одной модели в каждом — экспортируются через `kaggle_export_model.ipynb`).
2. Запусти **Шаг 1** → увидишь пути к моделям → впиши их в `MODEL_DIRS` (**Шаг 2**).
3. **Internet → Off**, **Accelerator → GPU T4** (НЕ P100).
4. Save & Run All → `submission.csv` в Output → **Submit**.

In [ ]:
# Шаг 1: посмотри, какие папки с моделями подключены (где лежит config.json).
# Подключи два датасета (по одной модели в каждом) через Input → + Add Input,
# затем скопируй нужные пути из вывода ниже в MODEL_DIRS (следующая ячейка).
import glob, os
print("доступные папки с моделями:")
for p in sorted(glob.glob("/kaggle/input/**/config.json", recursive=True)):
    print("   ", os.path.dirname(p))

In [ ]:
# Шаг 2: впиши сюда пути из вывода выше (две модели для ансамбля, или одну).
MODEL_DIRS = [
    "/kaggle/input/akkadian-byt5-full-seed13",   # <- замени на путь из вывода выше
    "/kaggle/input/akkadian-byt5-full-seed42",   # <- вторая модель (убери для одиночной)
]
NORMALIZE = True       # модели exp1+ учились на нормализованном входе
NUM_BEAMS = 4
CANDS_PER_MODEL = 4    # сколько beam-кандидатов брать с каждой модели в пул MBR
TEST = "/kaggle/input/competitions/deep-past-initiative-machine-translation/test.csv"
OUT = "/kaggle/working/submission.csv"

for d in MODEL_DIRS:
    assert os.path.isfile(os.path.join(d, "config.json")), f"нет config.json в {d}"
print("модели:", MODEL_DIRS)

In [ ]:
# --- вшитая нормализация (копия akkadian_nmt/normalize.py) ---
import re, unicodedata
_SUB = str.maketrans("₀₁₂₃₄₅₆₇₈₉ₓ", "0123456789x")
_DAMAGE = re.compile(r"[⸢⸣\[\]!?#*]")
_ANGLE = re.compile(r"<+[^<>]*>+")
_WS = re.compile(r"\s+")
GAP = "…"

def normalize_translit(text, keep_damage_marks=False):
    text = unicodedata.normalize("NFC", text).translate(_SUB)
    text = _ANGLE.sub(GAP, text)
    if not keep_damage_marks:
        text = _DAMAGE.sub("", text)
    text = re.sub(rf"(?:{GAP}\s*)+", GAP + " ", text)
    return _WS.sub(" ", text).strip()

# --- вшитый MBR-chrF (без sacrebleu): выбор консенсус-кандидата ---
from collections import Counter
def _ngrams(s, n):
    s = s.replace(" ", "")
    return Counter(s[i:i+n] for i in range(len(s)-n+1)) if len(s) >= n else Counter()
def _chrf(hyp, ref, nmax=6, beta=2.0):
    fs = []
    for n in range(1, nmax+1):
        h, r = _ngrams(hyp, n), _ngrams(ref, n)
        if not h or not r:
            continue
        ov = sum((h & r).values())
        p, rc = ov/sum(h.values()), ov/sum(r.values())
        fs.append(0.0 if p+rc == 0 else (1+beta**2)*p*rc/(beta**2*p+rc))
    return sum(fs)/len(fs) if fs else 0.0
def mbr_select(cands):
    cands = [c for c in cands if c.strip()] or cands
    if len(cands) == 1:
        return cands[0]
    best, bs = cands[0], -1.0
    for h in cands:
        s = sum(_chrf(h, o) for o in cands if o is not h) / (len(cands)-1)
        if s > bs:
            best, bs = h, s
    return best

In [ ]:
import torch, pandas as pd
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"
df = pd.read_csv(TEST)
texts = df["transliteration"].fillna("").tolist()
if NORMALIZE:
    texts = [normalize_translit(t) for t in texts]

pools = [[] for _ in texts]
for md in MODEL_DIRS:
    tok = AutoTokenizer.from_pretrained(md)
    model = AutoModelForSeq2SeqLM.from_pretrained(md).to(device).eval()
    with torch.inference_mode():
        for i in range(0, len(texts), 8):
            batch = texts[i:i+8]
            enc = tok(batch, return_tensors="pt", padding=True,
                      truncation=True, max_length=512).to(device)
            gen = model.generate(**enc, num_beams=NUM_BEAMS,
                                 num_return_sequences=CANDS_PER_MODEL, max_new_tokens=512)
            dec = tok.batch_decode(gen, skip_special_tokens=True)
            for j in range(len(batch)):
                pools[i+j].extend(dec[j*CANDS_PER_MODEL:(j+1)*CANDS_PER_MODEL])
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print("done", md)

hyps = [mbr_select(p) for p in pools]
pd.DataFrame({"id": df["id"], "translation": hyps}).to_csv(OUT, index=False)
print("wrote", OUT, "| rows:", len(hyps))
pd.read_csv(OUT).head()